In [ ]:
import pandas as pd
from db_queries import get_location_metadata
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from tqdm import tqdm
import numpy as np
from matplotlib.ticker import MaxNLocator
from pathlib import Path
import yaml
import xarray as xr

# Add FHS forecasted q0s to plots
import xarray as xr
import os

# Set paths ####################################################################
RAW_DATA_PATH = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/training_data/2026_09_13.01/data.parquet"
GBD_FILE = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/input/gbd_prevalence/gbd_mean_neonatal_mortality_prevalence.parquet"
FHS_PATH1 = "/mnt/share/forecasting/data/32/future/death/20260716_q0_reference_s130v98_nq0"  # Reference Q0 without ODA effects, pre-shifting to 2025 retro
FHS_PATH2 = "/mnt/share/forecasting/data/32/future/death/20260717_q0_reference_s130v108_noreagg_nq0"  # Reference q0 with ODA effects, pre-shifting to 2025 retro

FORMATTED_DIR = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/fhs_q0s"

# INFERENCE_FILE_09_14 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/results/2026_09_14.01/ssp245.parquet"  # Consumption merged on birth year
# RESULTS_09_14 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/models/2026_09_14.02/"  # .01 is the input spec; .02 is the fit

# RESULTS_09_21 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/models/2026_09_21.02/"
# RESULTS_09_22 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/models/2026_09_22.02/"
# RESULTS_09_23 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/models/2026_09_23.02/"  # birth_year dropped, consumption_median_RT rescaled to $/day
# RESULTS_09_24 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/models/2026_09_23.03/"  # birth_year dropped, consumption_median_RT, climate variables included
# RESULTS_09_25 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/models/2026_09_25.01/"  # final specification, sex_id modified
# RESULTS_09_28 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/models/2026_09_25.01/"  # final specification, sex_id modified


# INFERENCE_FILE_09_21 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/results/2026_09_22.01/ssp245.parquet"  # Inclusion of consumption_pd_national_median (logged)
# INFERENCE_FILE_09_22 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/results/2026_09_22.02/ssp245.parquet"  # Inclusion of consumption_pd_national_median (logged)
# INFERENCE_FILE_09_23 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/results/2026_09_23.01/ssp245.parquet"  # model 2026_09_23.02: consumption_median_RT (LDI based), no birth_year
# INFERENCE_FILE_09_24 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/results/2026_09_23.02/ssp245.parquet"  # model 2026_09_24: birth_year dropped, consumption_median_RT, climate variables included
# INFERENCE_FILE_09_25 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/results/2026_09_25.02/ssp245.parquet"  # final specification, sex_id modified
# INFERENCE_FILE_09_28 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/results/2026_09_28.01/ssp245.parquet"  # final specification, sex_id modified
INFERENCE_FILE_10_08 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/results/2026_10_08.01/ssp245.parquet"


PLOT_OUTPUT = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/plots/"


df_raw = pd.read_parquet(RAW_DATA_PATH)

# Add location names
locs = get_location_metadata(location_set_id=39, release_id=16)

df_raw = df_raw.merge(
    locs[["location_id", "location_name", "level", "super_region_name"]],
    on="location_id",
    how="left",
)

df_raw.head()

,year_start,year_end,nid,survey_name,int_year,int_month,sex_id,mothers_age_year,aod_months,age_month,...,year_id,age_group_id,int_birth_year_diff_months,log_consumption_pd,sdi,consumption_pd_national_median,consumption_median_RT,location_name,level,super_region_name
0,2009,2010,13109,DHS_MIS,None,None,2,None,0,0,...,2009,42,0,-0.245692,0.318291,2.179549,2.361256,Uganda,3,Sub-Saharan Africa
1,2022,2022,538795,None,2022,5,2,18,0,0,...,2021,42,11,1.312609,0.447416,3.220460,3.404259,United Republic of Tanzania,3,Sub-Saharan Africa
2,2017,2017,353526,None,2017,4,1,20,0,0,...,2017,42,0,2.448691,0.373539,3.087602,4.210878,Senegal,3,Sub-Saharan Africa
3,2019,2021,467681,DHS,None,None,1,None,0,0,...,2020,42,2,1.619791,0.583046,4.671741,5.111802,India,3,South Asia
4,2000,2000,19156,None,2000,2,2,24,0,0,...,1999,42,2,-0.227447,0.340183,1.772798,2.106530,Cambodia,3,"Southeast Asia, East Asia, and Oceania"


In [ ]:
year5q0_no_cuts = (
    (
        xr.open_dataarray(os.path.join(FHS_PATH1, "year5q0.nc"))
        .sel(scenario=130, sex_id=[1, 2])
        .mean(dim="draw")
        .squeeze()
    )
    .to_dataframe()
    .reset_index()
)
year5q0_with_cuts = (
    (
        xr.open_dataarray(os.path.join(FHS_PATH2, "year5q0.nc"))
        .sel(scenario=130, sex_id=[1, 2])
        .mean(dim="draw")
        .squeeze()
    )
    .to_dataframe()
    .reset_index()
)
day28q0_no_cuts = (
    (
        xr.open_dataarray(os.path.join(FHS_PATH1, "day28q0.nc"))
        .sel(scenario=130, sex_id=[1, 2])
        .mean(dim="draw")
        .squeeze()
    )
    .to_dataframe()
    .reset_index()
)
day28q0_with_cuts = (
    (
        xr.open_dataarray(os.path.join(FHS_PATH2, "day28q0.nc"))
        .sel(scenario=130, sex_id=[1, 2])
        .mean(dim="draw")
        .squeeze()
    )
    .to_dataframe()
    .reset_index()
)

In [13]:
# Two subsets of the residual step's prevalence plots ########################
# Reproduces what make_results_plots does, in this kernel, so it needs no
# rra_climate_health import:
#   1. read sev_means.parquet, clip the shifted columns to [0, 1] as the
#      diagnostics do
#   2. population-weight the most-detailed locations up the FHS hierarchy
#   3. draw one page per location, same line styles as the published pdf
# Then write two pdfs: Global + the 7 super regions, and the level-3 countries.
SUB_RV = "2026_09_28.01"
SUB_MEASURE = "neonatal_mortality"
SUB_ROOT = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition"
SUB_SEV = f"{SUB_ROOT}/{SUB_MEASURE}/results/{SUB_RV}/sev_means.parquet"
SUB_HIER = f"{SUB_ROOT}/input/fhs_hierarchy.parquet"
SUB_POP_FUT = "/mnt/share/forecasting/data/32/future/population/future_population_s130v41/population.nc"
SUB_POP_PAST = "/mnt/share/forecasting/data/16/past/population/20250603_etl_run_id_417/population.nc"
SUB_POP_SCENARIO = 130
SUB_REFERENCE = "ssp245"
SUB_ANCHOR = 2023  # constants.LAST_GBD_YEAR, where the shift is read off

# line styles copied from residual_diagnostics.PREVALENCE_STYLE_MAP / LABEL_MAP
SUB_STYLE = [
    ("gbd_value", "black", "-", "GBD Prevalence"),
    ("adjusted_constant_climate", "lightgreen", "-", "Shifted constant climate"),
    ("adjusted_ssp585", "purple", "-", "Shifted SSP 5-8.5"),
    ("adjusted_ssp126", "green", "-", "Shifted SSP 1-2.6"),
    ("model_value", "blue", ":", "Original model"),
    ("residual_model_value", "blue", "--", "Residual model"),
    ("adjusted_model_value", "blue", "-", "Shifted model"),
]

sub_res = pd.read_parquet(SUB_SEV)
sub_hier = pd.read_parquet(SUB_HIER)
_ages = sorted(sub_res.index.get_level_values("age_group_id").unique())
_sexes = sorted(sub_res.index.get_level_values("sex_id").unique())
_years = sorted(sub_res.index.get_level_values("year_id").unique())

# the clipping make_results_plots applies before aggregating
for _c in [c for c in sub_res.columns if c.startswith("adjusted_")]:
    sub_res[_c] = sub_res[_c].clip(0, 1)


# Reported age groups that the population files store only as their parts,
# mirroring constants.AGE_GROUP_AGGREGATES. Neonatal's 42 is early + late
# neonatal, so the population has to be summed over [2, 3] first.
SUB_AGE_AGGREGATES = {
    4: [388, 389],
    5: [238, 34],
    42: [2, 3],
    1: [2, 3, 388, 389, 238, 34],
}


def sub_load_population():
    """Past + future population means, on the sev_means demographic grid."""
    detailed = sorted({a for age in _ages for a in SUB_AGE_AGGREGATES.get(age, [age])})
    fut = (
        xr.open_dataset(SUB_POP_FUT)
        .sel(age_group_id=detailed, sex_id=_sexes, scenario=SUB_POP_SCENARIO)
        .to_dataframe()
        .reset_index()
    )
    past = (
        xr.open_dataset(SUB_POP_PAST)
        .sel(age_group_id=detailed, sex_id=_sexes)
        .to_dataframe()
        .reset_index()
    )
    out = []
    for df in (past, fut):
        val = "population" if "population" in df.columns else "draws"
        keys = ["location_id", "year_id", "age_group_id", "sex_id"]
        df = (
            df.groupby(keys, as_index=False)[val]
            .mean()
            .rename(columns={val: "population"})
        )
        # roll the detailed groups back up into the reported ones
        parts = []
        for age in _ages:
            members = SUB_AGE_AGGREGATES.get(age, [age])
            sub = df[df.age_group_id.isin(members)].copy()
            sub["age_group_id"] = age
            parts.append(sub.groupby(keys, as_index=False)["population"].sum())
        out.append(pd.concat(parts))
    pop = pd.concat(out)
    # past and future overlap; the forecast file wins, matching the pipeline
    pop = pop.drop_duplicates(
        ["location_id", "year_id", "age_group_id", "sex_id"], keep="last"
    )
    return pop[pop.year_id.isin(_years)]


sub_pop = sub_load_population()

# FHS forecast q0s, for reference against the pipeline's own lines. These files
# already carry every level of the hierarchy - Global, super regions, regions,
# countries and subnationals - at age_group_id 42, so they merge straight on
# without any aggregation of their own. Values are plain probabilities
# (rate_denominator = 1), the same scale as gbd_value, so they only need x100.
SUB_FHS = {
    "fhs_with_cuts": (
        "/mnt/share/forecasting/data/32/future/death/"
        "20260717_q0_reference_s130v108_noreagg_nq0",
        "#d1495b",
        "-",
        "FHS forecasts with ODA cuts",
    ),
    "fhs_no_cuts": (
        "/mnt/share/forecasting/data/32/future/death/"
        "20260716_q0_reference_s130v98_nq0",
        "#d1495b",
        "--",
        "FHS forecasts without ODA cuts",
    ),
}


def sub_load_fhs():
    """Draw-averaged day28q0 for both ODA scenarios, keyed like sub_agg."""
    out = None
    for col, (path, *_rest) in SUB_FHS.items():
        da = xr.open_dataarray(f"{path}/day28q0.nc")
        df = (
            da.sel(scenario=130, sex_id=_sexes)
            .mean(dim="draw")
            .squeeze()
            .to_dataframe()
            .reset_index()
            .rename(columns={"value": col})
        )
        keys = ["location_id", "year_id", "sex_id", "age_group_id"]
        df = df[keys + [col]]
        out = df if out is None else out.merge(df, on=keys, how="outer")
    return out


sub_fhs = sub_load_fhs()
print(
    f"FHS overlay: {sub_fhs.location_id.nunique()} locations, "
    f"{int(sub_fhs.year_id.min())}-{int(sub_fhs.year_id.max())}, "
    f"age {sorted(sub_fhs.age_group_id.unique())}"
)

# population-weight every most-detailed location up its path_to_top_parent
_detailed = sub_hier[sub_hier["most_detailed"] == 1][
    ["location_id", "path_to_top_parent"]
]
_vals = [c for c, *_ in SUB_STYLE if c in sub_res.columns]
_flat = (
    sub_res.reset_index()
    .merge(_detailed, on="location_id", how="inner")
    .merge(
        sub_pop, on=["location_id", "year_id", "age_group_id", "sex_id"], how="inner"
    )
)
_flat["agg_id"] = _flat["path_to_top_parent"].str.split(",")
_exp = _flat.explode("agg_id")
_exp["agg_id"] = _exp["agg_id"].astype(int)
for _c in _vals:
    _exp[_c] = _exp[_c] * _exp["population"]
_keys = ["agg_id", "year_id", "age_group_id", "sex_id"]
sub_agg = _exp.groupby(_keys, as_index=False)[_vals + ["population"]].sum(min_count=1)
for _c in _vals:
    sub_agg[_c] = sub_agg[_c] / sub_agg["population"]
sub_agg = sub_agg.rename(columns={"agg_id": "location_id"})

# Raw survey data, to overlay on the country pages. It is coded to country
# level and by BIRTH year, plotted against the model's year axis; sex_id is a
# string in the training extract so it needs coercing to match.
SUB_MIN_BIRTHS = 100
sub_obs = (
    df_raw.assign(sex_id=lambda d: pd.to_numeric(d["sex_id"], errors="coerce"))
    .dropna(subset=["sex_id", "birth_year", "neonatal_mortality"])
    .groupby(["location_id", "birth_year", "sex_id"], observed=True)
    .agg(rate=("neonatal_mortality", "mean"), n=("neonatal_mortality", "size"))
    .reset_index()
    .query("n >= @SUB_MIN_BIRTHS")
)
sub_obs["sex_id"] = sub_obs["sex_id"].astype(int)
print(
    f"survey overlay: {len(sub_obs):,} country/year/sex cells with "
    f">= {SUB_MIN_BIRTHS} births, {sub_obs.location_id.nunique()} countries"
)

_lv = sub_hier.set_index("location_id")["level"]
_nm = sub_hier.set_index("location_id")["location_name"]
print(
    f"aggregated to {sub_agg.location_id.nunique()} locations "
    f"from {_flat.location_id.nunique()} most-detailed inputs; "
    f"levels {sorted(set(_lv.reindex(sub_agg.location_id.unique()).dropna().astype(int)))}"
)


def sub_make_pdf(loc_ids, out_path, title_note, *, overlay_survey=False):
    """One page per location: a sex-by-age grid, styled like the published pdf."""
    with PdfPages(out_path) as pdf:
        for loc_id in loc_ids:
            fig, axes = plt.subplots(
                len(_sexes),
                len(_ages),
                figsize=(14, 8),
                sharex=True,
                sharey=True,
                squeeze=False,
            )
            handles, labels = [], []
            for j, sex_id in enumerate(_sexes):
                for i, age_id in enumerate(_ages):
                    ax = axes[j, i]
                    blk = sub_agg[
                        (sub_agg.location_id == loc_id)
                        & (sub_agg.sex_id == sex_id)
                        & (sub_agg.age_group_id == age_id)
                    ].sort_values("year_id")
                    if overlay_survey:
                        # behind the lines; point size scales with births so
                        # thin survey years read as less reliable
                        o = sub_obs[
                            (sub_obs.location_id == loc_id) & (sub_obs.sex_id == sex_id)
                        ]
                        if len(o):
                            ax.scatter(
                                o.birth_year,
                                o.rate,
                                s=np.clip(o.n / 60, 4, 60),
                                color="0.45",
                                alpha=0.55,
                                linewidths=0,
                                zorder=0,
                                label=f"survey data (size = births, "
                                f"min {SUB_MIN_BIRTHS})",
                            )
                    for col, colour, ls, lab in SUB_STYLE:
                        if col in blk.columns:
                            ax.plot(
                                blk.year_id, blk[col], color=colour, ls=ls, label=lab
                            )
                    # FHS reference forecasts; 2024-2050 only, so these appear
                    # as a segment rather than spanning the panel
                    f = sub_fhs[
                        (sub_fhs.location_id == loc_id)
                        & (sub_fhs.sex_id == sex_id)
                        & (sub_fhs.age_group_id == age_id)
                    ].sort_values("year_id")
                    for col, (_pth, colour, ls, lab) in SUB_FHS.items():
                        if col in f.columns and f[col].notna().any():
                            ax.plot(
                                f.year_id,
                                f[col],
                                color=colour,
                                ls=ls,
                                lw=1.8,
                                zorder=5,
                                label=lab,
                            )
                    # the anchor year: the shifted model meets GBD here by
                    # construction, and GBD itself stops here
                    ax.axvline(
                        SUB_ANCHOR,
                        color="0.45",
                        lw=1.1,
                        ls="-.",
                        zorder=1,
                        label=f"anchor year {SUB_ANCHOR}",
                    )
                    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
                    if j == 0:
                        ax.set_title(f"Age group {age_id}")
                    ax.set_ylabel("Male" if sex_id == 1 else "Female")
                    if i > 0:
                        ax.set_ylabel("")
                        ax.tick_params(labelleft=False)
                    if j == len(_sexes) - 1:
                        ax.set_xlabel("Year")
                    if not handles:
                        handles, labels = ax.get_legend_handles_labels()
            fig.suptitle(
                f"{SUB_MEASURE.capitalize()} Prevalence - Location {loc_id}: "
                f"{_nm.get(loc_id)}",
                fontsize=16,
            )
            fig.legend(
                handles, labels, loc="lower center", ncol=3, bbox_to_anchor=(0.5, 0)
            )
            fig.tight_layout(rect=[0, 0.05, 1, 1])
            pdf.savefig(fig)
            plt.close(fig)
    print(f"Saved {out_path} ({len(loc_ids)} pages, {title_note})")


_ordered = [l for l in sub_agg.location_id.unique() if l in _lv.index]
# The survey data is coded to country level, so it goes on the country pages
# only; putting a national rate on a super-region panel would mix grains.
SUB_SETS = {
    "global_superregions": (
        [l for l in _ordered if _lv[l] in (0, 1)],
        "Global and the super regions",
        False,
    ),
    "countries": ([l for l in _ordered if _lv[l] == 3], "level-3 countries", True),
}
for _tag, (_locs, _note, _ov) in SUB_SETS.items():
    sub_make_pdf(
        _locs,
        f"{PLOT_OUTPUT}{SUB_MEASURE}_prevalence_plots_{_tag}.pdf",
        _note,
        overlay_survey=_ov,
    )

FHS overlay: 538 locations, 2024-2050, age [42]
survey overlay: 1,681 country/year/sex cells with >= 100 births, 60 countries
aggregated to 513 locations from 473 most-detailed inputs; levels [0, 1, 2, 3, 4]
Saved /mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/plots/neonatal_mortality_prevalence_plots_global_superregions.pdf (8 pages, Global and the super regions)
Saved /mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/plots/neonatal_mortality_prevalence_plots_countries.pdf (204 pages, level-3 countries)


In [12]:
day28q0_with_cuts

,year_id,location_id,sex_id,scenario,age_group_id,value
0,2024,1,1,130,42,0.017830
1,2024,1,2,130,42,0.013963
2,2024,4,1,130,42,0.007803
3,2024,4,2,130,42,0.006075
4,2024,5,1,130,42,0.001760
...,...,...,...,...,...,...
29047,2050,60908,2,130,42,0.010712
29048,2050,94364,1,130,42,0.024611
29049,2050,94364,2,130,42,0.012406
29050,2050,95069,1,130,42,0.022524
